# 03｜4Pi-Confocal 与 Richards-Wolf 参考解工作流（中文）

本 notebook 目标：
1. 演示 4Pi 相位扫描；
2. 演示双臂不平衡与像差；
3. 用 Debye 参考解做小网格对照。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pyotf.microscope import FourPiConfocalMicroscope
from pyotf.vector import richards_wolf_linearly_polarized_psf

## 第 1 步：创建 4Pi 模型

In [ ]:
base_kwargs = dict(model="hanser", na=1.1, ni=1.33, wl=520, size=64, pixel_size=80, oversample_factor=1)

model_4pi = FourPiConfocalMicroscope(
    wl_exc=488,
    pinhole_size=0.8,
    phase_exc=0.0,
    phase_det=0.0,
    amp_ratio_exc=0.9,
    amp_ratio_det=1.1,
    aberrations_exc_arm2={"vertical coma": 0.12},
    aberrations_det_arm1={"oblique astigmatism": 0.08},
    **base_kwargs
)

print("PSF shape:", model_4pi.PSF.shape)
print("PSF sum:", model_4pi.PSF.sum())

## 第 2 步：做相位扫描（phase_scan）

观察相位变化对轴向响应的影响。

In [ ]:
phases = np.linspace(0, 2*np.pi, 7)
stack = model_4pi.phase_scan(phases, channel="both")
print("stack shape:", stack.shape)

plt.figure(figsize=(7,4))
for i, p in enumerate(phases):
    prof = stack[i].sum(axis=(1,2))
    plt.plot(prof, label=f"{p:.2f}")
plt.title("4Pi 相位扫描：轴向积分曲线")
plt.xlabel("z index")
plt.ylabel("integrated intensity")
plt.legend(ncol=2, fontsize=8)
plt.show()

## 第 3 步：Richards-Wolf 参考解（小网格）

注意：这个计算会比 FFT 路线慢很多。建议先用小网格。

In [ ]:
x = np.linspace(-200, 200, 21)
y = np.linspace(-200, 200, 21)
z = np.linspace(-300, 300, 11)

psf_rw = richards_wolf_linearly_polarized_psf(
    wl=520, na=1.1, ni=1.33, x=x, y=y, z=z, n_theta=40, n_phi=64
)
print("RW PSF shape:", psf_rw.shape)
print("RW PSF sum:", psf_rw.sum())

## 第 4 步：看参考解中心切片

In [ ]:
z0 = psf_rw.shape[0] // 2
plt.figure(figsize=(5,4))
plt.imshow(psf_rw[z0], cmap="viridis")
plt.title("Richards-Wolf 参考解中心切片")
plt.colorbar()
plt.show()

## 练习建议

1. 把 `n_theta` / `n_phi` 调大，观察耗时和图像稳定性；
2. 给 `richards_wolf_linearly_polarized_psf` 增加 `pcoefs`，观察像差效应；
3. 对比 4Pi 模型与参考解的轴向 profile 趋势。